# Exp2 → Exp3 Prediction (Quick Run)

This notebook cell uses Exp2 Pass@1 to predict Exp3 (until-correct) success and expected attempts,
using the baseline formulas (q = 1 - (1 - p)^k, A = [1 - (1 - p)^k] / p). Finite-population correction
is optional and controlled via POOL_SIZE.


In [3]:
# Configuration
RESULTS_DIR = './results/'            # root of results tree
K = 3                               # Exp3: max attempts per type (set to your k)
POOL_SIZE = None                     # None => i.i.d. baseline; set integer N for finite-pool correction
PROVIDER = 'openai'                      # e.g., 'openai' or None for all
MODEL = 'openai/gpt-5'                         # e.g., 'openai/gpt-5' or None for all
CALIBRATE = False                    # set True to fit simple calibration if Exp3 observations exist

In [ ]:
from exp2_to_exp3_predict import (
    predict_q_from_exp2, predict_A_from_exp2,
    calibrate_logit_linear, apply_calibration
)
from visualize_results import CAPTCHAVisualizer
import pandas as pd
import numpy as np

viz = CAPTCHAVisualizer(results_dir=RESULTS_DIR)
if viz.data.empty:
    raise RuntimeError('No results found under RESULTS_DIR')

df = viz.data.copy()
if PROVIDER:
    df = df[df['provider'] == PROVIDER]
if MODEL:
    df = df[df['provider_model'] == MODEL]

# Exp2 summary
exp2 = df[df['experiment'] == 'exp2'].copy()
if exp2.empty:
    raise RuntimeError('Exp2 results not found in current selection')
if 'n' not in exp2.columns:
    exp2['n'] = 1
exp2_grp = exp2.groupby(
    ['provider','model','provider_model','task_type'],
    as_index=False
).agg(
    n=('n','max'),
    p_hat=('pass','mean')
)

# Exp3 observations (optional validation)
exp3 = df[df['experiment'] == 'exp3'].copy()
if not exp3.empty:
    exp3_obs = exp3.groupby(
        ['provider','model','provider_model','task_type'],
        as_index=False
    ).agg(
        q_obs=('pass','mean'),
        A_obs=('avg_attempts','mean')
    )
else:
    exp3_obs = pd.DataFrame(
        columns=['provider','model','provider_model','task_type','q_obs','A_obs']
    )

rows = []
for _, r in exp2_grp.iterrows():
    n = int(r['n'])
    p_hat = float(r['p_hat'])

    # Difficulty parameter λ = -ln(1 - p), clip p to avoid log(0)
    p_clipped = float(np.clip(p_hat, 1e-9, 1.0 - 1e-9))
    lam = -np.log(1.0 - p_clipped)

    # Baseline i.i.d. form: q_iid(k) = 1 - exp(-k λ)
    q_iid = 1.0 - np.exp(-K * lam)

    # Final prediction (includes finite-pool correction if POOL_SIZE is set)
    q_pred = predict_q_from_exp2(p_hat, n, K, N_pool=POOL_SIZE)
    A_pred = predict_A_from_exp2(p_hat, n, K, N_pool=POOL_SIZE)

    rows.append([
        r['provider'], r['model'], r['provider_model'], r['task_type'],
        n, p_hat, lam, q_iid, q_pred, A_pred
    ])

pred_df = pd.DataFrame(
    rows,
    columns=[
        'provider','model','provider_model','task_type',
        'n','p_hat','lambda','q_iid','q_pred','A_pred'
    ]
)
merged = pred_df.merge(
    exp3_obs,
    on=['provider','model','provider_model','task_type'],
    how='left'
)

if CALIBRATE and not exp3_obs.empty:
    valid = merged.dropna(subset=['q_pred','q_obs'])
    if len(valid) >= 3:
        a, b = calibrate_logit_linear(
            valid['q_pred'].to_numpy(), valid['q_obs'].to_numpy()
        )
        merged['q_pred_cal'] = apply_calibration(
            merged['q_pred'].to_numpy(), a, b
        )
        print(f'Calibration fitted: a={a:.3f}, b={b:.3f}')
    else:
        print('Not enough pairs to calibrate; skipping')

display(merged.sort_values(['provider_model','task_type']).head(20))
merged.to_csv('./exp2_to_exp3_predictions_nb.csv', index=False)
print('Saved to ./exp2_to_exp3_predictions_nb.csv')
